In [1]:
import nflreadpy as nfl
import pandas as pd

In [7]:
import nflreadpy as nfl
import pandas as pd

seasons = [2022, 2023, 2024, 2025]

#Pull team and schedule data from the nflverse repositories through nflreadpy
team_stats = nfl.load_team_stats(seasons = seasons).to_pandas()
schedules = nfl.load_schedules(seasons = seasons).to_pandas()

schedules['id'] = schedules['season'].astype(str) + '_' + schedules['week'].astype(str) + '_' + schedules['home_team'] + '_' + schedules['away_team']

#Self-joins the team_stats so home_team and away_team stats are included in each row
df = pd.merge(team_stats, team_stats, left_on = ['season', 'week', 'team'], right_on = ['season', 'week', 'opponent_team'], suffixes = ['_home', '_away'])

df['id'] = df['season'].astype(str) + '_' + df['week'].astype(str) + '_' + df['team_home'] + '_' + df['team_away']

#Sets the columns of interest from the schedule dataframe
schedule_cols = ['id', 'home_score', 'away_score', 'home_team', 'away_team', 'overtime', 'home_rest', 'away_rest']

#Joins the stats with schedule data, in order to fetch scores and determine which team is the home team
df = pd.merge(df, schedules[schedule_cols], how = 'inner', on = 'id')

#Drops redundant columns created by the self-joni
df = df.drop(['team_home', 'opponent_team_home', 'team_away', 'season_type_away', 'opponent_team_away', 'game_id_home', 'game_id_away'], axis = 1)
#Removes the _home suffix from all columns that are the same for each team
df = df.rename(columns = {'season_type_home': 'season_type', 'home_rest': 'rest_home', 'away_rest': 'rest_away'})

#Creates the target column, home_win, which is 1 if the home team won
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

#Displays the first few rows of the dataframe
with pd.option_context('display.max_columns', 12):
    display(df.head(20))

print(df.shape)

df.to_csv('data/nfl_data.csv', index = False)

,season,week,season_type,completions_home,attempts_home,passing_yards_home,...,home_team,away_team,overtime,rest_home,rest_away,home_win
0,2022,1,REG,24,38,205,...,ARI,KC,0,7,7,0
1,2022,1,REG,20,33,215,...,ATL,NO,0,7,7,0
2,2022,1,REG,16,27,235,...,CAR,CLE,0,7,7,0
3,2022,1,REG,8,17,121,...,CHI,SF,0,7,7,1
4,2022,1,REG,33,53,338,...,CIN,PIT,1,7,7,0
5,2022,1,REG,21,42,198,...,DAL,TB,0,7,7,0
6,2022,1,REG,21,37,215,...,DET,PHI,0,7,7,0
7,2022,1,REG,23,37,240,...,HOU,IND,1,7,7,0
8,2022,1,REG,29,41,240,...,LA,BUF,0,7,7,0
9,2022,1,REG,26,34,279,...,LAC,LV,0,7,7,1


(1139, 206)


Data Dictionary: https://nflreadr.nflverse.com/articles/dictionary_team_stats.html